# ============================================================================
# GEMINI AGENT - Decision making and orchestration
# ============================================================================

In [ ]:
class AnomalyDetectionAgent:
    """AI Agent powered by Gemini 2.0 Flash for intelligent decision-making"""
    
    def __init__(self, api_key: str):
        genai.configure(api_key=api_key)
        self.model = genai.GenerativeModel('gemini-2.0-flash-exp')
        self.tools = AnomalyDetectionTools()
        self.conversation_history = []
        self.execution_log = []
        
    def _call_gemini(self, prompt: str) -> str:
        """Call Gemini with conversation context"""
        response = self.model.generate_content(prompt)
        return response.text
    
    def _log_operation(self, operation: str, details: Dict):
        """Log all operations for final summary"""
        self.execution_log.append({
            "operation": operation,
            "details": details,
            "timestamp": pd.Timestamp.now().isoformat()
        })
    
    def interact(self, message: str) -> str:
        """Main interaction method - agent's brain"""
        system_context = """You are an Anomaly Detection Master AI Agent. Your role is to:
        1. Guide users through anomaly detection model selection
        2. Validate inputs and suggest corrections
        3. Make intelligent decisions about which tools to call
        4. Cross-check results and decide if re-runs are needed
        5. Provide expert insights on model performance
        
        You have access to these LOCAL tools (no data goes to you):
        - load_dataset: Load and validate CSV files
        - create_time_splits: Create 5 out-of-time validation splits
        - calculate_separation_distance: Compute separation metric
        - train_and_evaluate: Train models and evaluate performance
        - get_hyperparameter_grid: Get predefined hyperparameter combinations
        
        IMPORTANT: You only receive metadata, never raw data. All processing is local.
        """
        
        full_prompt = f"{system_context}\n\nUser: {message}\n\nRespond with your decision and any actions to take."
        response = self._call_gemini(full_prompt)
        return response
    
    def validate_file_path(self, filepath: str) -> Tuple[bool, str]:
        """Validate and suggest corrections for file paths"""
        if not filepath:
            suggestion = "Please provide a file path. Example: 'train_data.csv'"
            return False, suggestion
        
        if not filepath.endswith('.csv'):
            suggestion = f"File should be CSV format. Did you mean '{filepath}.csv'?"
            return False, suggestion
        
        if not os.path.exists(filepath):
            # Check for common mistakes
            suggestions = []
            if '\\' in filepath:
                suggestions.append(f"Try with forward slashes: {filepath.replace(chr(92), '/')}")
            
            current_dir = os.listdir('.')
            csv_files = [f for f in current_dir if f.endswith('.csv')]
            if csv_files:
                suggestions.append(f"CSV files in current directory: {', '.join(csv_files)}")
            
            return False, f"File not found. {' '.join(suggestions)}"
        
        return True, "File path is valid"
    
    def run_pipeline(self):
        """Main agent pipeline"""
        print("="*80)
        print("🤖 ANOMALY DETECTION AI AGENT ACTIVATED")
        print("Powered by Gemini 2.0 Flash")
        print("="*80)
        print()
        
        # Step 1: Load Training Dataset
        print("📊 STEP 1: Loading Training Dataset")
        print("-" * 80)
        
        while True:
            train_path = input("Enter training dataset path (CSV): ").strip()
            valid, message = self.validate_file_path(train_path)
            if not valid:
                print(f"❌ {message}")
                retry = input("Try again? (yes/no): ").strip().lower()
                if retry != 'yes':
                    return
                continue
            
            result = self.tools.load_dataset(train_path)
            if not result.get("success", True):
                
                print(f"❌ Error loading file: {result.get('error')}")
                print("\n💡 Agent suggestion: Check file format and ensure it's a valid CSV")
                retry = input("Try again? (yes/no): ").strip().lower()
                if retry != 'yes':
                    return
                continue
            
            train_data = result["data"]
            metadata = result["metadata"]
            
            print(f"✅ Training data loaded successfully!")
            print(f"   Rows: {metadata['rows']}")
            print(f"   Columns: {len(metadata['columns'])}")
            print(f"   Numeric features: {len(metadata['numeric_columns'])}")
            
            self._log_operation("load_train_data", metadata)
            break
        
        # Step 2: Load Test Dataset
        print("\n📊 STEP 2: Loading Test Dataset")
        print("-" * 80)
        
        while True:
            test_path = input("Enter test dataset path (CSV): ").strip()
            valid, message = self.validate_file_path(test_path)
            
            if not valid:
                print(f"❌ {message}")
                retry = input("Try again? (yes/no): ").strip().lower()
                if retry != 'yes':
                    return
                continue
            
            result = self.tools.load_dataset(test_path)
            
            if not result.get("success", True):
                print(f"❌ Error loading file: {result.get('error')}")
                retry = input("Try again? (yes/no): ").strip().lower()
                if retry != 'yes':
                    return
                continue
            
            test_data = result["data"]
            test_metadata = result["metadata"]
            
            print(f"✅ Test data loaded successfully!")
            print(f"   Rows: {test_metadata['rows']}")
            
            self._log_operation("load_test_data", test_metadata)
            break
        
        # Step 3: Feature Selection
        print("\n🎯 STEP 3: Feature Selection")
        print("-" * 80)
        print(f"Available numeric columns: {metadata['numeric_columns']}")
        
        agent_prompt = f"""
        The dataset has these numeric columns: {metadata['numeric_columns']}
        Total columns: {metadata['columns']}
        
        Should we use all numeric columns as features, or should the user specify?
        Provide a recommendation and ask the user to confirm.
        """
        
        agent_response = self._call_gemini(agent_prompt)
        print(f"\n🤖 Agent: {agent_response}")
        
        feature_input = input("\nEnter feature columns (comma-separated) or 'all' for all numeric: ").strip()
        
        if feature_input.lower() == 'all':
            feature_cols = metadata['numeric_columns']
        else:
            feature_cols = [c.strip() for c in feature_input.split(',')]
            # Validate features
            invalid = [f for f in feature_cols if f not in metadata['columns']]
            if invalid:
                print(f"❌ Invalid columns: {invalid}")
                print("Using all numeric columns instead.")
                feature_cols = metadata['numeric_columns']
        
        print(f"✅ Selected features ({len(feature_cols)}): {feature_cols}")
        self._log_operation("feature_selection", {"features": feature_cols})
        
        # Step 4: Create Time Splits
        print("\n📈 STEP 4: Creating Out-of-Time Validation Splits")
        print("-" * 80)
        
        splits = self.tools.create_time_splits(train_data, n_splits=5)
        print(f"✅ Created 5 out-of-time splits (80:20 train:val)")
        for i, (train, val) in enumerate(splits):
            print(f"   Fold {i+1}: Train={len(train)}, Val={len(val)}")
        
        self._log_operation("create_splits", {"n_splits": 5, "split_ratio": "80:20"})
        
        # Step 5: Model Training and Evaluation
        print("\n🚀 STEP 5: Training and Evaluating Models")
        print("-" * 80)
        
        hyperparameter_grid = self.tools.get_hyperparameter_grid()
        results = []
        
        total_experiments = sum(len(params) for params in hyperparameter_grid.values())
        print(f"Total experiments to run: {total_experiments}")
        print()
        
        experiment_count = 0
        
        for algorithm, param_list in hyperparameter_grid.items():
            print(f"\n🔍 Testing {algorithm}...")
            
            for param_idx, params in enumerate(param_list):
                experiment_count += 1
                print(f"   [{experiment_count}/{total_experiments}] Params: {params}")
                
                result = self.tools.train_and_evaluate(
                    algorithm, params, splits, feature_cols
                )
                
                if result["success"]:
                    avg_dist = result["avg_separation_distance"]
                    print(f"   ✅ Avg Separation Distance: {avg_dist:.4f}")
                    
                    results.append({
                        "Algorithm": algorithm,
                        "Hyperparameters": json.dumps(params),
                        "Separation_Distance": avg_dist,
                        "Fold_Distances": result["fold_distances"]
                    })
                else:
                    print(f"   ❌ Failed: {result.get('errors', [])}")
        
        # Step 6: Results Analysis
        print("\n📊 STEP 6: Results Analysis")
        print("=" * 80)
        
        results_df = pd.DataFrame(results)
        results_df = results_df.sort_values("Separation_Distance", ascending=False)
        
        print("\n🏆 MODEL PERFORMANCE RANKING")
        print("-" * 80)
        print(results_df[["Algorithm", "Hyperparameters", "Separation_Distance"]].to_string(index=False))
        
        # Step 7: Agent Decision on Results Quality
        print("\n🤖 STEP 7: Agent Quality Assessment")
        print("-" * 80)
        
        best_distance = results_df.iloc[0]["Separation_Distance"]
        worst_distance = results_df.iloc[-1]["Separation_Distance"]
        
        quality_prompt = f"""
        Analyze these anomaly detection results:
        - Best separation distance: {best_distance:.4f}
        - Worst separation distance: {worst_distance:.4f}
        - Total models tested: {len(results_df)}
        - Best algorithm: {results_df.iloc[0]['Algorithm']}
        
        Questions:
        1. Is the best separation distance acceptable? (>1.0 is typically good)
        2. Is there enough variance in results to trust the ranking?
        3. Should we re-run with different hyperparameters or all results look reliable?
        
        Provide your expert assessment.
        """
        
        agent_assessment = self._call_gemini(quality_prompt)
        print(f"🤖 Agent Assessment:\n{agent_assessment}")
        
        rerun = input("\nShould we re-run any experiments? (yes/no): ").strip().lower()
        
        if rerun == 'yes':
            print("💡 Re-run functionality: Modify hyperparameter_grid and restart pipeline")
        
        # Step 8: Final Summary
        print("\n" + "=" * 80)
        print("📋 FINAL SUMMARY")
        print("=" * 80)
        
        best_model = results_df.iloc[0]
        
        summary_prompt = f"""
        Generate a comprehensive summary of this anomaly detection experiment:
        
        Operations executed: {len(self.execution_log)}
        Best performing model: {best_model['Algorithm']}
        Best hyperparameters: {best_model['Hyperparameters']}
        Best separation distance: {best_model['Separation_Distance']:.4f}
        
        Explain:
        1. Why this model performed best
        2. What the separation distance tells us about model quality
        3. Key insights from the hyperparameter choices
        4. Recommendations for production deployment
        
        Be concise but insightful.
        """
        
        final_summary = self._call_gemini(summary_prompt)
        
        print(f"\n🤖 AGENT'S FINAL ANALYSIS:")
        print("-" * 80)
        print(final_summary)
        
        print("\n" + "=" * 80)
        print("✅ ANOMALY DETECTION PIPELINE COMPLETED")
        print("=" * 80)
        
        # Save results
        output_file = "anomaly_detection_results.csv"
        results_df.to_csv(output_file, index=False)
        print(f"\n💾 Results saved to: {output_file}")
        
        # Save execution log
        log_file = "execution_log.json"
        with open(log_file, 'w') as f:
            json.dump(self.execution_log, f, indent=2)
        print(f"📝 Execution log saved to: {log_file}")